# 🕰️ 01 — The Three Clocks

One late record. Three clocks: **event time**, **Delta commit time**, and **query time**.

The goal is brutally simple: prove that an old event timestamp does **not** mean the row existed in an old table version.

In [ ]:
# --- Lab configuration -------------------------------------------------------
# Defaults to the current catalog/schema. Override if you want a dedicated lab area.
CATALOG = spark.catalog.currentCatalog()
SCHEMA = spark.catalog.currentDatabase()
DATA_TABLE = "morlock_customer_state"
EVIDENCE_TABLE = "morlock_time_travel_evidence"
EXPERIMENT_ID = "morlock-time-travel-001"

spark.conf.set("spark.sql.session.timeZone", "UTC")

def qi(x: str) -> str:
    return f"`{x.replace('`', '``')}`"

def fqn(catalog: str, schema: str, table: str) -> str:
    return ".".join(qi(x) for x in (catalog, schema, table))

DATA_FQN = fqn(CATALOG, SCHEMA, DATA_TABLE)
EVIDENCE_FQN = fqn(CATALOG, SCHEMA, EVIDENCE_TABLE)

print("Data table    :", DATA_FQN)
print("Evidence table:", EVIDENCE_FQN)

In [ ]:
from pyspark.sql import functions as F, types as T
from datetime import datetime, timezone
import hashlib, json

BUSINESS_COLUMNS = [
    "customer_id", "name", "segment", "balance",
    "status", "event_time", "source_system"
]

EVIDENCE_SCHEMA = T.StructType([
    T.StructField("experiment_id", T.StringType(), False),
    T.StructField("operation_id", T.LongType(), False),
    T.StructField("operation_name", T.StringType(), False),
    T.StructField("wall_clock_timestamp", T.TimestampType(), False),
    T.StructField("event_time_min", T.TimestampType(), True),
    T.StructField("event_time_max", T.TimestampType(), True),
    T.StructField("delta_version", T.LongType(), True),
    T.StructField("delta_commit_timestamp", T.TimestampType(), True),
    T.StructField("delta_operation", T.StringType(), True),
    T.StructField("converted_delta_version", T.LongType(), True),
    T.StructField("converted_delta_timestamp", T.TimestampType(), True),
    T.StructField("iceberg_snapshot_id", T.StringType(), True),
    T.StructField("iceberg_metadata_path", T.StringType(), True),
    T.StructField("hudi_instant_or_identifier", T.StringType(), True),
    T.StructField("row_count", T.LongType(), True),
    T.StructField("logical_checksum", T.StringType(), True),
    T.StructField("schema_json", T.StringType(), True),
    T.StructField("schema_fingerprint", T.StringType(), True),
    T.StructField("active_file_count", T.LongType(), True),
    T.StructField("active_file_paths_hash", T.StringType(), True),
    T.StructField("runtime_version", T.StringType(), True),
    T.StructField("spark_version", T.StringType(), True),
    T.StructField("table_properties_json", T.StringType(), True),
    T.StructField("notes", T.StringType(), True),
])

def logical_checksum(df, columns=BUSINESS_COLUMNS):
    """Deterministic checksum for this intentionally tiny lab table."""
    row_json = F.to_json(
        F.struct(*[F.col(c) for c in columns]),
        options={"ignoreNullFields": "false"},
    )
    return (
        df.select(F.sha2(row_json, 256).alias("_h"))
          .agg(F.sha2(F.concat_ws("||", F.sort_array(F.collect_list("_h"))), 256).alias("checksum"))
          .first()["checksum"]
    )

def schema_signature(df):
    raw = df.schema.json()
    return raw, hashlib.sha256(raw.encode()).hexdigest()

def observed_active_files(df):
    paths = sorted(
        r["path"] for r in
        df.select(F.input_file_name().alias("path"))
          .where(F.col("path") != "")
          .distinct().collect()
    )
    return len(paths), hashlib.sha256("\n".join(paths).encode()).hexdigest()

def table_properties_json(table_fqn):
    props = {r["key"]: r["value"] for r in spark.sql(f"SHOW TBLPROPERTIES {table_fqn}").collect()}
    return json.dumps(props, sort_keys=True)

def latest_history(table_fqn):
    return spark.sql(f"DESCRIBE HISTORY {table_fqn} LIMIT 1").first()

def history_for_version(table_fqn, version):
    rows = spark.sql(f"DESCRIBE HISTORY {table_fqn}").where(F.col("version") == int(version)).collect()
    if not rows:
        raise ValueError(f"Version {version} not found")
    return rows[0]

def runtime_version():
    try:
        return spark.conf.get("spark.databricks.clusterUsageTags.sparkVersion")
    except Exception:
        return "unknown"

def next_operation_id():
    row = spark.table(EVIDENCE_FQN).agg(F.max("operation_id").alias("m")).first()
    return 1 if row["m"] is None else int(row["m"]) + 1

def record_evidence(operation_name, notes=""):
    df = spark.table(DATA_FQN)
    hist = latest_history(DATA_FQN)
    bounds = df.agg(F.min("event_time").alias("lo"), F.max("event_time").alias("hi")).first()
    schema_json, schema_hash = schema_signature(df)
    file_count, file_hash = observed_active_files(df)

    row = {
        "experiment_id": EXPERIMENT_ID,
        "operation_id": next_operation_id(),
        "operation_name": operation_name,
        "wall_clock_timestamp": datetime.now(timezone.utc).replace(tzinfo=None),
        "event_time_min": bounds["lo"],
        "event_time_max": bounds["hi"],
        "delta_version": int(hist["version"]),
        "delta_commit_timestamp": hist["timestamp"],
        "delta_operation": hist["operation"],
        "converted_delta_version": None,
        "converted_delta_timestamp": None,
        "iceberg_snapshot_id": None,
        "iceberg_metadata_path": None,
        "hudi_instant_or_identifier": None,
        "row_count": df.count(),
        "logical_checksum": logical_checksum(df),
        "schema_json": schema_json,
        "schema_fingerprint": schema_hash,
        "active_file_count": file_count,
        "active_file_paths_hash": file_hash,
        "runtime_version": runtime_version(),
        "spark_version": spark.version,
        "table_properties_json": table_properties_json(DATA_FQN),
        "notes": notes,
    }
    out = spark.createDataFrame([row], EVIDENCE_SCHEMA)
    out.write.format("delta").mode("append").saveAsTable(EVIDENCE_FQN)
    return out

## 1. Recover the baseline from persisted evidence

In [ ]:
assert spark.catalog.tableExists(f"{CATALOG}.{SCHEMA}.{DATA_TABLE}"), "Run notebook 00 first."
assert spark.catalog.tableExists(f"{CATALOG}.{SCHEMA}.{EVIDENCE_TABLE}"), "Run notebook 00 first."

baseline = (spark.table(EVIDENCE_FQN)
    .where((F.col("experiment_id") == EXPERIMENT_ID) & (F.col("operation_name") == "initial_load"))
    .orderBy(F.col("operation_id").desc()).first())

assert baseline is not None
BASELINE_VERSION = int(baseline["delta_version"])
print("Baseline version:", BASELINE_VERSION)

## 2. Create one deliberately late record

In [ ]:
LATE_CUSTOMER_ID = 999
late_rows = [(999, "Larry Latecomer", "time_traveller", 42.42, "ACTIVE", "2026-08-10T09:15:00Z", "delayed_morlock_bus")]
schema = "customer_id LONG, name STRING, segment STRING, balance DOUBLE, status STRING, event_time STRING, source_system STRING"
late_df = spark.createDataFrame(late_rows, schema=schema).withColumn("event_time", F.to_timestamp("event_time"))
LATE_EVENT_TIME = late_df.first()["event_time"]
display(late_df)

## 3. Prove the baseline does not contain Larry

In [ ]:
n = spark.sql(f"""
  SELECT count(*) AS n
  FROM {DATA_FQN} VERSION AS OF {BASELINE_VERSION}
  WHERE customer_id = {LATE_CUSTOMER_ID}
""").first()["n"]
assert n == 0
print(f"✅ Version {BASELINE_VERSION}: Larry does not exist yet.")

## 4. Append the old event now

In [ ]:
late_df.write.format("delta").mode("append").saveAsTable(DATA_FQN)

late_evidence = record_evidence(
    "late_record_append",
    "Customer 999 has event_time 2026-08-10T09:15:00Z but enters Delta only in this append."
)
display(late_evidence)

## 5. Find the first Delta version that contains the row

In [ ]:
latest_version = int(latest_history(DATA_FQN)["version"])
checks = []
first_visible_version = None

for v in range(BASELINE_VERSION, latest_version + 1):
    n = spark.sql(f"""
      SELECT count(*) AS n
      FROM {DATA_FQN} VERSION AS OF {v}
      WHERE customer_id = {LATE_CUSTOMER_ID}
    """).first()["n"]
    checks.append((v, int(n)))
    if n > 0 and first_visible_version is None:
        first_visible_version = v

assert first_visible_version is not None
display(spark.createDataFrame(checks, "delta_version LONG, late_record_count LONG"))
print("First visible Delta version:", first_visible_version)

## 6. Put the three clocks side by side

In [ ]:
first_hist = history_for_version(DATA_FQN, first_visible_version)
FIRST_VISIBLE_COMMIT_TS = first_hist["timestamp"]
QUERY_TIME = datetime.now(timezone.utc).replace(tzinfo=None)

clocks = spark.createDataFrame([{
    "customer_id": LATE_CUSTOMER_ID,
    "event_time": LATE_EVENT_TIME,
    "first_visible_delta_version": first_visible_version,
    "first_visible_commit_timestamp": FIRST_VISIBLE_COMMIT_TS,
    "query_time": QUERY_TIME,
}])
display(clocks)

## 7. Execute the claim

In [ ]:
assert first_visible_version > BASELINE_VERSION
assert LATE_EVENT_TIME < FIRST_VISIBLE_COMMIT_TS

before = spark.sql(f"""
  SELECT count(*) AS n
  FROM {DATA_FQN} VERSION AS OF {first_visible_version - 1}
  WHERE customer_id = {LATE_CUSTOMER_ID}
""").first()["n"]

at_first = spark.sql(f"""
  SELECT count(*) AS n
  FROM {DATA_FQN} VERSION AS OF {first_visible_version}
  WHERE customer_id = {LATE_CUSTOMER_ID}
""").first()["n"]

assert before == 0
assert at_first == 1

print("✅ PROVED")
print("   Event time              :", LATE_EVENT_TIME, "UTC")
print("   Baseline version        :", BASELINE_VERSION, "-> absent")
print("   First containing version:", first_visible_version, "-> present")
print("   First commit timestamp  :", FIRST_VISIBLE_COMMIT_TS, "UTC")
print("   Query time              :", QUERY_TIME, "UTC")

## 8. Article screenshot

In [ ]:
proof = spark.createDataFrame([{
    "claim": "old event != old table state",
    "event_time": LATE_EVENT_TIME,
    "baseline_version": BASELINE_VERSION,
    "present_in_baseline": False,
    "first_version_containing_row": first_visible_version,
    "first_commit_containing_row": FIRST_VISIBLE_COMMIT_TS,
    "event_precedes_first_commit": LATE_EVENT_TIME < FIRST_VISIBLE_COMMIT_TS,
}])
display(proof)

## 9. Evidence ledger

In [ ]:
display(
    spark.table(EVIDENCE_FQN)
         .where(F.col("experiment_id") == EXPERIMENT_ID)
         .orderBy("operation_id")
)

## Result

Larry's event says **August 10**. Delta history says Larry did not exist in the baseline and first appeared in a later committed version.

**Event time describes the business event. Delta version/commit time describes table history. They are not interchangeable.**